In [ ]:
# import libraries
import pandas as pd
import plotly.graph_objects as go

In [ ]:
# load data
companies = pd.read_csv("../data/european_producers_locations.csv", sep=";")
species = pd.read_csv("../data/european_producers_species.csv", sep=";")

In [ ]:
# preview companies
companies.head()

In [ ]:
# preview species
species.head()

In [ ]:
# split data on production and method
species["production"] = [method.split("_")[1] for method in species["ID_Prod_Meth"]]
species["method"] = [method.split("_")[2] for method in species["ID_Prod_Meth"]]

In [ ]:
# select macroalgae producers
good_ids = [row["ID_Site"] for _, row in species.iterrows() if row["production"] == "Macroalgae"]
mask_companies = companies["ID_Site"].isin(good_ids)
companies = companies.loc[mask_companies, :].reset_index(drop=True)
mask_species = species["ID_Site"].isin(good_ids)
species = species.loc[mask_species, :].reset_index(drop=True)

In [ ]:
# add method info to companies
companies["method"] = ""
for i in range(companies.shape[0]):
    mask_id = species["ID_Site"] == companies.loc[i, "ID_Site"]
    method_current = species.loc[mask_id, "method"].unique()
    if len(method_current) == 1:
        companies.loc[i, "method"] = method_current
    else:
        print(i)
companies["method"] = companies["method"].astype("str")

In [ ]:
# discriminate between harvesting and aquaculture and translate in French
companies["method_cat"] = ""
for i in range(companies.shape[0]):
    method_current = companies.loc[i, "method"]
    if ("Harvesting" in method_current) & ("Aquaculture" in method_current):
        companies.loc[i, "method_cat"] = "Récolte & Culture"
    elif "Harvesting" in method_current:
        companies.loc[i, "method_cat"] = "Récolte"
    else:
        companies.loc[i, "method_cat"] = "Culture"

In [ ]:
# format numbers from strings to float
columns = ["Lat", "Long"]
for column in columns:
    companies.loc[:, column] = [strnb.replace(",", ".") for strnb in companies.loc[:, column]]
companies[columns] = companies[columns].astype("float")

In [ ]:
# plot map of producers
fig=go.Figure()

methods = companies["method_cat"].unique()
colors=["#5194E8", "#0950AD", "#CCE9FF"]
for i in range(len(methods)):
    mask_method = companies["method_cat"] == methods[i]
    fig.add_trace(
        go.Scattermap(
            lon = companies.loc[mask_method, "Long"],
            lat = companies.loc[mask_method, "Lat"],
            mode='markers',
            name=methods[i],
            marker=dict(
                size=10,
                color=colors[i],
            ),
            hovertemplate=companies.loc[mask_method, "Owner_name"] + "<extra></extra>"
        )
    )

fig.update_layout(
    plot_bgcolor="#F9BF6B",
    paper_bgcolor="#F9BF6B",
    hovermode="closest",
    hoverlabel=dict(
        font_size=14,
        font_family="Montserrat",
        font_color="#FDF2EE",
        bordercolor="#FDF2EE",
    ),
    hoverdistance=100,
    map=dict(
        bearing=0,
        center=dict(
            lat=54,
            lon=2
        ),
        zoom=2,
        style="basic"
    ),
    modebar=dict(
        orientation="v"
    ),
    legend=dict(
        orientation="h",
        font_family="Montserrat",
        font_size=13,
        font_color="#113972",
        bgcolor="rgba(0,0,0,0)",
        entrywidth=180,
        yanchor="top",
        y=0.0,
        xanchor="center",
        x=0.5
    ),
    margin=dict(
        l=30,
        r=30,
        t=30,
        b=30
    ),
    height=500,
    width=750
)

fig.show()

In [ ]:
fig.write_html("../figures/european_producers_pc.html", include_plotlyjs="cdn")

In [ ]:
# plot map of producers
fig=go.Figure()

methods = companies["method_cat"].unique()
colors=["#5194E8", "#0950AD", "#CCE9FF"]
for i in range(len(methods)):
    mask_method = companies["method_cat"] == methods[i]
    fig.add_trace(
        go.Scattermap(
            lon = companies.loc[mask_method, "Long"],
            lat = companies.loc[mask_method, "Lat"],
            mode='markers',
            name=methods[i],
            marker=dict(
                size=7,
                color=colors[i],
            ),
            hovertemplate=companies.loc[mask_method, "Owner_name"] + "<extra></extra>"
        )
    )

fig.update_layout(
    plot_bgcolor="#F9BF6B",
    paper_bgcolor="#F9BF6B",
    hovermode="closest",
    hoverlabel=dict(
        font_size=14,
        font_family="Montserrat",
        font_color="#FDF2EE",
        bordercolor="#FDF2EE",
    ),
    hoverdistance=100,
    map=dict(
        bearing=0,
        center=dict(
            lat=44,
            lon=2
        ),
        zoom=1,
        style="basic"
    ),
    modebar=dict(
        orientation="v"
    ),
    legend=dict(
        orientation="h",
        font_family="Montserrat",
        font_size=13,
        font_color="#113972",
        bgcolor="rgba(0,0,0,0)",
        entrywidth=180,
        yanchor="top",
        y=0.0,
        xanchor="center",
        x=0.5
    ),
    margin=dict(
        l=30,
        r=30,
        t=30,
        b=30
    ),
    height=400,
    width=250
)

fig.show()

In [ ]:
fig.write_html("../figures/european_producers_mobile.html", include_plotlyjs="cdn")